# Module 3 — 01: Load Data & Exploratory Data Analysis — Heart Disease Datasets

This notebook loads three heart-disease-related datasets that are already available locally under `data/raw/` and explores each one in turn: **Dataset 1** (2022 BRFSS "Personal Key Indicators of Heart Disease"), **Dataset 2** (Heart Failure Prediction), and **Dataset 3** (2015 BRFSS "Heart Disease Health Indicators").

## Dataset 1: Personal Key Indicators of Heart Disease (2022)

### 1. Load Data

In [1]:
import pandas as pd

data_path = '../data/raw/dataset1/heart_2022_with_nans.csv'
df = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df.shape)
df.head()

Loaded file: ../data/raw/dataset1/heart_2022_with_nans.csv
Shape: (445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


**Result:** The dataset is loaded directly from the local `data/raw/dataset1/` folder — no download step is needed. We load **`heart_2022_with_nans.csv`** on purpose, since it lets us demonstrate a realistic EDA and preprocessing workflow that includes handling missing values, rather than starting from an already-cleaned file. The loaded dataframe has **445,132 rows and 40 columns**, matching the official 2022 BRFSS update of this dataset.

### 2. Exploratory Data Analysis (EDA)

In [2]:
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])
print()
print('Data types:')
print(df.dtypes.value_counts())
print()
df.info()

Number of rows: 445132
Number of columns: 40

Data types:
str        34
float64     6
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 445132 entries, 0 to 445131
Data columns (total 40 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   State                      445132 non-null  str    
 1   Sex                        445132 non-null  str    
 2   GeneralHealth              443934 non-null  str    
 3   PhysicalHealthDays         434205 non-null  float64
 4   MentalHealthDays           436065 non-null  float64
 5   LastCheckupTime            436824 non-null  str    
 6   PhysicalActivities         444039 non-null  str    
 7   SleepHours                 439679 non-null  float64
 8   RemovedTeeth               433772 non-null  str    
 9   HadHeartAttack             442067 non-null  str    
 10  HadAngina                  440727 non-null  str    
 11  HadStroke                  443575 non-nul

**Result:** The dataset has **445,132 rows and 40 columns**, made up of **34 categorical (`object`) columns** (e.g. `State`, `Sex`, `GeneralHealth`, most of the Yes/No health-history flags such as `HadHeartAttack`, `HadDiabetes`, `HadStroke`) and **6 numeric (`float64`) columns** (`PhysicalHealthDays`, `MentalHealthDays`, `SleepHours`, `HeightInMeters`, `WeightInKilograms`, `BMI`). Every column has some missing values — the `Non-Null Count` for each column is consistently below 445,132 — which confirms this is the raw survey file and that missing-value handling will be a necessary preprocessing step. The dataframe uses about 135.8+ MB of memory in its current (unoptimized) form.

In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print('Number of columns with missing values:', len(missing_summary))
print(missing_summary.to_string())

Number of columns with missing values: 38
                           missing_count  missing_pct
TetanusLast10Tdap                  82516        18.54
PneumoVaxEver                      77040        17.31
HIVTesting                         66127        14.86
ChestScan                          56046        12.59
CovidPos                           50764        11.40
HighRiskLastYear                   50623        11.37
BMI                                48806        10.96
FluVaxLast12                       47121        10.59
AlcoholDrinkers                    46574        10.46
WeightInKilograms                  42078         9.45
ECigaretteUsage                    35660         8.01
SmokerStatus                       35462         7.97
HeightInMeters                     28652         6.44
DifficultyErrands                  25656         5.76
DifficultyConcentrating            24240         5.45
DifficultyWalking                  24012         5.39
DifficultyDressingBathing          23915

**Result:** **38 of the 40 columns** contain at least one missing value — only `State` and `Sex` are fully populated. The most affected columns are vaccination/screening-history fields that many respondents simply skip or don't know: `TetanusLast10Tdap` (18.54%), `PneumoVaxEver` (17.31%), `HIVTesting` (14.86%), `ChestScan` (12.59%), `CovidPos` (11.40%), and `HighRiskLastYear` (11.37%). The core numeric health metrics have moderate missingness (`BMI` 10.96%, `WeightInKilograms` 9.45%, `HeightInMeters` 6.44%), while the disease-history Yes/No flags (`HadHeartAttack`, `HadStroke`, `HadDiabetes`, etc.) and core demographics are the most complete, at well under 1% missing. This pattern is typical of self-reported survey data: harder-to-recall or less commonly asked questions have more gaps. It tells us we can safely impute the low-missingness columns with simple statistics, while columns above ~10% missing deserve a bit more care during preprocessing.

In [4]:
numeric_cols = ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']
df[numeric_cols].describe().round(2)

,PhysicalHealthDays,MentalHealthDays,SleepHours,HeightInMeters,WeightInKilograms,BMI
count,434205.00,436065.00,439679.00,416480.00,403054.00,396326.00
mean,4.35,4.38,7.02,1.70,83.07,28.53
std,8.69,8.39,1.50,0.11,21.45,6.55
min,0.00,0.00,1.00,0.91,22.68,12.02
25%,0.00,0.00,6.00,1.63,68.04,24.13
50%,0.00,0.00,7.00,1.70,80.74,27.44
75%,3.00,5.00,8.00,1.78,95.25,31.75
max,30.00,30.00,24.00,2.41,292.57,99.64


**Result:** The numeric columns show sensible but skewed distributions typical of health survey data. `PhysicalHealthDays` and `MentalHealthDays` (days out of the past 30 with poor health) both have a median of 0–4 but a mean around 4.3–4.4, indicating a right-skewed distribution where most people report very few bad days while a smaller group reports many (up to the maximum of 30). `SleepHours` is centered close to the recommended 7–8 hours (mean 7.02, IQR 6–8), though the max of 24 is very likely a data-entry outlier. `HeightInMeters` (mean 1.70m) and `WeightInKilograms` (mean 83.07kg) look reasonable, but `BMI` has a very wide range (12.02 to 99.64) with a mean of 28.53 — already in the "overweight" clinical range — and the extreme maximum values suggest a handful of outliers that should be kept in mind (though not necessarily removed) during preprocessing.

In [5]:
import matplotlib.pyplot as plt

target_counts = df['HadHeartAttack'].value_counts(dropna=False)
target_pct = df['HadHeartAttack'].value_counts(normalize=True, dropna=False).mul(100).round(2)
print('HadHeartAttack value counts:')
print(target_counts)
print()
print('HadHeartAttack percentages:')
print(target_pct)

fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind='bar', color=['#4C72B0', '#DD8452', '#999999'], ax=ax)
ax.set_title('Distribution of HadHeartAttack (target variable)')
ax.set_xlabel('HadHeartAttack')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

**Result:** The target variable `HadHeartAttack` is heavily imbalanced: **93.67% (416,959)** respondents answered "No", only **5.64% (25,108)** answered "Yes", and a small **0.69% (3,065)** are missing. This roughly 17:1 ratio between the majority and minority classes is expected for a real-world disease-prevalence variable, but it has an important consequence for modeling later: metrics like plain accuracy would be misleading (a model that always predicts "No" would already score ~94%), so techniques such as class weighting, resampling, or threshold tuning — combined with metrics like recall, F1, or ROC-AUC — will matter more than raw accuracy. The missing target rows will be dropped in preprocessing, since we cannot train or evaluate on a label we don't have.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cat_cols_to_plot = ['Sex', 'GeneralHealth', 'AgeCategory', 'SmokerStatus']
for ax, col in zip(axes.flat, cat_cols_to_plot):
    df[col].value_counts(dropna=False).plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Result:** The survey population is fairly balanced by sex (roughly 52% Female / 48% Male) and skews toward older respondents — the largest single age bracket is "65 to 69", and counts generally decline for younger brackets, which is typical of BRFSS telephone-survey data. Self-reported general health is mostly positive: "Very good" and "Good" together account for the large majority of responses, with only a small share reporting "Poor" health. For smoking, the majority of respondents (`Never smoked`) have never smoked, roughly a quarter are former smokers, and current smokers (daily or occasional) make up a relatively small minority. These distributions confirm the dataset represents a broad, mostly older, general-population sample rather than a clinically pre-selected group, which is appropriate for a population health-risk study.

In [ ]:
corr = df[numeric_cols].corr()
print(corr.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='black', fontsize=8)
ax.set_title('Correlation matrix (numeric features)')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Result:** The correlation matrix shows that WeightInKilograms and BMI are strongly correlated (0.86), which is expected since BMI is mathematically derived from weight (and height). HeightInMeters and WeightInKilograms show a moderate positive correlation (0.47), while HeightInMeters and BMI are nearly uncorrelated (-0.03), confirming that BMI successfully normalizes out the effect of height. PhysicalHealthDays and MentalHealthDays are moderately correlated (0.32), suggesting that people who report more days of poor physical health also tend to report more days of poor mental health. SleepHours is weakly (and mostly negatively) correlated with all other numeric features, indicating it behaves largely independently. Overall, aside from the expected Weight-BMI relationship, no pair of features is so highly correlated that multicollinearity is a major concern for this dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.boxplot(column='BMI', by='HadHeartAttack', ax=axes[0])
axes[0].set_title('BMI by HadHeartAttack')
axes[0].set_xlabel('HadHeartAttack')
axes[0].set_ylabel('BMI')
health_order = ['Excellent', 'Very good', 'Good', 'Fair', 'Poor']
rate = df.groupby('GeneralHealth')['HadHeartAttack'].apply(lambda x: (x == 'Yes').mean() * 100)
rate = rate.reindex(health_order)
rate.plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Heart Attack Rate (%) by GeneralHealth')
axes[1].set_xlabel('GeneralHealth')
axes[1].set_ylabel('Heart Attack Rate (%)')
plt.suptitle('')
plt.tight_layout()
plt.show()
print(rate.round(2))

**Result:** The boxplot shows that BMI distributions are very similar between people who did and did not have a heart attack (both medians around 27-28), so BMI alone is a weak discriminator for this target. In contrast, the bar chart of heart attack rate by self-reported GeneralHealth shows a clear, strong, monotonic relationship: the rate rises from 1.53% for 'Excellent' health to 2.84% ('Very good'), 5.78% ('Good'), 11.99% ('Fair'), and 21.08% for 'Poor' health. This indicates GeneralHealth is a much more informative predictor of heart attack risk than BMI, and highlights the value of including categorical self-reported health features in the model alongside numeric measurements.

## Dataset 2: Heart Failure Prediction

### 1. Load Data

In [ ]:
data_path = '../data/raw/dataset2/heart.csv'
df2 = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df2.shape)
df2.head()

**Result:** Loaded from the local `data/raw/dataset2/` folder. This is the [fedesoriano Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) dataset, a much smaller and already mostly-clean table: **918 rows and 12 columns**, combining 5 clinical/demographic categorical fields (`Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`) with 6 numeric measurements (`Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`) and the binary target `HeartDisease`.

### 2. Exploratory Data Analysis (EDA)

In [ ]:
print('Number of rows:', df2.shape[0])
print('Number of columns:', df2.shape[1])
print()
print('Data types:')
print(df2.dtypes.value_counts())
print()
df2.info()

**Result:** The dataset has **918 rows and 12 columns** — much smaller than Dataset 1 — made up of **6 numeric columns** (5 `int64` + 1 `float64`, `Oldpeak`) and **5 categorical columns** (`Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`), plus the integer target `HeartDisease`. Every column has a full 918 non-null entries, so there are no NaN gaps at all in this dataset — unlike Dataset 1's raw survey data. Total memory usage is only about 325 KB, reflecting the much smaller row count.

In [ ]:
missing2 = df2.isna().sum()
print('Total missing values:', missing2.sum())
print(missing2)

**Result:** There are **zero missing values** across all 918 rows and 12 columns — this dataset was already cleaned before publication. However, as we'll see next, "zero missing values" does not mean the data has no data-quality issues: some numeric columns use `0` as a placeholder for a measurement that wasn't taken, rather than leaving it truly blank.

In [ ]:
numeric_cols2 = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
df2[numeric_cols2].describe().round(2)

**Result:** `Age` ranges from 28 to 77 (mean 53.5), and `MaxHR` (mean 136.8) and `Oldpeak` (ST depression, mean 0.89, can be negative) look reasonable. `FastingBS` is really a binary flag (0/1) despite its numeric dtype, with a mean of 0.23 (~23% of patients have fasting blood sugar > 120 mg/dl). The columns worth a closer look are `RestingBP` (min **0**) and `Cholesterol` (min **0**) — a resting blood pressure or cholesterol reading of exactly 0 is not physiologically possible, so these are very likely placeholder values for "not measured" rather than genuine readings.

In [ ]:
print('RestingBP == 0:', (df2['RestingBP'] == 0).sum(), 'rows')
print('Cholesterol == 0:', (df2['Cholesterol'] == 0).sum(), 'rows',
      f"({(df2['Cholesterol'] == 0).mean() * 100:.1f}% of rows)")

**Result:** Exactly **1 row** has `RestingBP == 0`, and **172 rows (18.7%)** have `Cholesterol == 0` — far too many to be genuine zero-cholesterol readings. These are treated as **missing-value placeholders** rather than real measurements: in preprocessing, both columns are converted from 0 to NaN before median-imputation, so the model doesn't learn from physiologically impossible zero values.

In [ ]:
target_counts2 = df2['HeartDisease'].value_counts()
target_pct2 = df2['HeartDisease'].value_counts(normalize=True).mul(100).round(2)
print('HeartDisease value counts:')
print(target_counts2)
print()
print('HeartDisease percentages:')
print(target_pct2)

fig, ax = plt.subplots(figsize=(5, 4))
target_counts2.plot(kind='bar', color=['#DD8452', '#4C72B0'], ax=ax)
ax.set_title('Distribution of HeartDisease (target variable)')
ax.set_xlabel('HeartDisease')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Result:** Unlike Dataset 1, the target here is **not** heavily imbalanced: **508 patients (55.34%)** have heart disease (`HeartDisease = 1`) and **410 (44.66%)** do not. This near-balanced split is typical of curated clinical datasets assembled specifically for heart-disease classification (as opposed to Dataset 1's general-population survey), and means plain accuracy is a much more reasonable metric here than it was for Dataset 1.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cat_cols_to_plot2 = ['Sex', 'ChestPainType', 'RestingECG', 'ST_Slope']
for ax, col in zip(axes.flat, cat_cols_to_plot2):
    df2[col].value_counts().plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Result:** The dataset skews heavily male (`Sex`: 725 M / 193 F, ~79% male), a known characteristic of this particular clinical cohort. `ChestPainType` is dominated by `ASY` (asymptomatic, 496 patients) — notably the category most associated with heart disease, as seen below. `RestingECG` is mostly `Normal`, with `ST` and `LVH` abnormalities less common. `ST_Slope` splits mainly between `Flat` (460) and `Up` (395), with `Down` being rare (63).

In [ ]:
corr2 = df2[numeric_cols2].corr()
print(corr2.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr2, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols2)))
ax.set_yticks(range(len(numeric_cols2)))
ax.set_xticklabels(numeric_cols2, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols2)
for i in range(len(numeric_cols2)):
    for j in range(len(numeric_cols2)):
        ax.text(j, i, f'{corr2.iloc[i, j]:.2f}', ha='center', va='center', color='black', fontsize=8)
ax.set_title('Correlation matrix (numeric features)')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Result:** No pair of numeric features is strongly correlated. The strongest relationships are `Age`↔`MaxHR` (-0.38, older patients tend to have a lower max heart rate) and `Age`↔`Oldpeak` (0.26). `Cholesterol` is weakly negatively correlated with `FastingBS` (-0.26) — consistent with the many placeholder-zero `Cholesterol` values noted earlier diluting the true relationship. Overall, multicollinearity is not a concern for this feature set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rate_cp = (df2.groupby('ChestPainType')['HeartDisease'].mean() * 100).round(2)
rate_cp.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Heart Disease Rate (%) by ChestPainType')
axes[0].set_ylabel('Heart Disease Rate (%)')

rate_slope = (df2.groupby('ST_Slope')['HeartDisease'].mean() * 100).round(2)
rate_slope.plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Heart Disease Rate (%) by ST_Slope')
axes[1].set_ylabel('Heart Disease Rate (%)')
plt.tight_layout()
plt.show()
print(rate_cp)
print(rate_slope)

**Result:** Both features are strong predictors. Patients with **asymptomatic** chest pain (`ASY`) have a heart disease rate of **79.03%**, far higher than `ATA` (13.87%), `NAP` (35.47%), or `TA` (43.48%) — somewhat counterintuitively, the *absence* of typical chest-pain symptoms is the strongest single warning sign in this dataset. Similarly, a **Flat** (82.83%) or **Down** (77.78%) ST segment slope during exercise is associated with a much higher heart disease rate than an **Up**-sloping ST segment (19.75%). Both `ChestPainType` and `ST_Slope` look like they will be highly informative features for a classification model.

## Dataset 3: Heart Disease Health Indicators (BRFSS 2015)

### 1. Load Data

In [ ]:
data_path = '../data/raw/dataset3/heart_disease_health_indicators_BRFSS2015.csv'
df3 = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df3.shape)
df3.head()

**Result:** Loaded from the local `data/raw/dataset3/` folder. This is the [alexteboul Heart Disease Health Indicators](https://www.kaggle.com/datasets/alexteboul/heart-disease-health-indicators-dataset) dataset, derived from the 2015 BRFSS survey: **253,680 rows and 22 columns**, all already numerically encoded (binary flags, ordinal scales, and a few continuous measurements), with the binary target `HeartDiseaseorAttack`.

### 2. Exploratory Data Analysis (EDA)

In [ ]:
print('Number of rows:', df3.shape[0])
print('Number of columns:', df3.shape[1])
print()
print('Data types:')
print(df3.dtypes.value_counts())
print()
df3.info()

**Result:** The dataset has **253,680 rows and 22 columns**, all stored as `float64` — including columns that are conceptually binary flags (e.g. `HighBP`, `Smoker`, `Sex`) or small ordinal scales (`GenHlth` 1–5, `Age` 1–13 brackets, `Education` 1–6, `Income` 1–8), reflecting that this file was already fully pre-processed/encoded by its publisher before release. Memory usage is about 44.6 MB — by far the largest of the three datasets by row count, though each row is comparatively lightweight since every feature is already numeric.

In [ ]:
print('Total missing values:', df3.isna().sum().sum())
print(df3.isna().sum())

**Result:** There are **zero missing values** anywhere in the dataset — every one of the 253,680 rows is fully populated across all 22 columns. This is the cleanest of the three datasets in terms of completeness, so no imputation will actually change any values here (though the pipeline still runs the same imputation step for consistency with the other datasets).

In [ ]:
numeric_cols3 = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
df3[numeric_cols3].describe().round(2)

**Result:** `BMI` ranges from 12 to 98 (mean 28.4), already in the "overweight" clinical range on average, similar to Dataset 1. `MentHlth` and `PhysHlth` (days of poor mental/physical health out of the past 30) are both heavily right-skewed: median 0 but a mean around 3.2–4.2, meaning most respondents report zero bad days while a minority report many. `GenHlth` (self-reported general health, 1=Excellent to 5=Poor), `Age` (1–13 age brackets), `Education` (1–6) and `Income` (1–8) are all small ordinal scales rather than continuous measurements, so their means (e.g. `GenHlth` mean 2.51) should be read as a position on that scale rather than a physical unit.

In [ ]:
dup_count = df3.duplicated().sum()
print('Duplicate rows:', dup_count)
print(f'Duplicate rows (%): {dup_count / len(df3) * 100:.2f}%')

**Result:** **23,899 rows (9.42%)** are exact duplicates of another row across all 22 columns. This is plausible here — unlike Dataset 1's 40 mostly free-form columns, every one of these 22 features is binary or a small discrete scale, so two different respondents can easily share an identical combination of answers by chance. Still, since these are meant to represent distinct survey respondents, exact duplicate rows are dropped during preprocessing so the model doesn't implicitly over-weight whichever response pattern happens to repeat most.

In [ ]:
target_counts3 = df3['HeartDiseaseorAttack'].value_counts()
target_pct3 = df3['HeartDiseaseorAttack'].value_counts(normalize=True).mul(100).round(2)
print('HeartDiseaseorAttack value counts:')
print(target_counts3)
print()
print('HeartDiseaseorAttack percentages:')
print(target_pct3)

fig, ax = plt.subplots(figsize=(5, 4))
target_counts3.plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax)
ax.set_title('Distribution of HeartDiseaseorAttack (target variable)')
ax.set_xlabel('HeartDiseaseorAttack')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Result:** The target is heavily imbalanced, similar to Dataset 1: **229,787 respondents (90.58%)** report no heart disease/attack history, while **23,893 (9.42%)** do — roughly a **1:9.6** minority-to-majority ratio. As with Dataset 1, this means plain accuracy would be misleading for evaluating a classifier trained on this data, and techniques like class weighting, resampling, or threshold tuning will matter more than raw accuracy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cat_cols_to_plot3 = ['HighBP', 'Smoker', 'PhysActivity', 'Sex']
for ax, col in zip(axes.flat, cat_cols_to_plot3):
    df3[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Result:** `HighBP` is fairly evenly split (144,851 without vs. 108,829 with high blood pressure). `Smoker` is similarly split (141,257 vs. 112,423 who report having smoked at least 100 cigarettes in their lifetime). `PhysActivity` skews toward respondents who report physical activity in the past 30 days (191,920 vs. 61,760 who don't). `Sex` is coded 0=Female/1=Male, with slightly more female respondents (141,974 vs. 111,706) — consistent with typical BRFSS telephone-survey demographics.

In [ ]:
corr3 = df3[numeric_cols3].corr()
print(corr3.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr3, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols3)))
ax.set_yticks(range(len(numeric_cols3)))
ax.set_xticklabels(numeric_cols3, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols3)
for i in range(len(numeric_cols3)):
    for j in range(len(numeric_cols3)):
        ax.text(j, i, f'{corr3.iloc[i, j]:.2f}', ha='center', va='center', color='black', fontsize=8)
ax.set_title('Correlation matrix (numeric features)')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Result:** `PhysHlth` and `GenHlth` are the most correlated pair (0.52), which makes sense — more days of poor physical health tend to coincide with worse self-reported general health. `MentHlth` and `PhysHlth` are moderately correlated (0.35). `Education` and `Income` are the strongest correlation overall (0.45), a common socioeconomic pattern. No pair reaches a level that would indicate a multicollinearity problem for modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rate_genhlth = (df3.groupby('GenHlth')['HeartDiseaseorAttack'].mean() * 100).round(2)
rate_genhlth.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Heart Disease/Attack Rate (%) by GenHlth')
axes[0].set_xlabel('GenHlth (1=Excellent .. 5=Poor)')
axes[0].set_ylabel('Rate (%)')

rate_highbp = (df3.groupby('HighBP')['HeartDiseaseorAttack'].mean() * 100).round(2)
rate_highbp.plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Heart Disease/Attack Rate (%) by HighBP')
axes[1].set_xlabel('HighBP (0=No, 1=Yes)')
axes[1].set_ylabel('Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(rate_genhlth)
print(rate_highbp)

**Result:** Both features show a clear, strong relationship with the target, echoing the `GeneralHealth` finding from Dataset 1. The heart disease/attack rate rises monotonically with `GenHlth` from **2.24%** ('Excellent') to **4.63%**, **10.46%**, **21.31%**, and **34.00%** ('Poor') — a pattern remarkably consistent with Dataset 1's GeneralHealth chart, despite these being two independent BRFSS survey years. `HighBP` is also strongly associated: **16.47%** of respondents with high blood pressure report heart disease/attack vs. only **4.12%** of those without, making it one of the most informative single binary features in this dataset.